In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from delta.tables import DeltaTable
from pyspark.sql.functions import current_timestamp

In [0]:
def get_additional_columns(all_columns: list, primary_keys: list) -> list:
    """
    Get the list of additional columns to be considered for updates.
    Exclude:
    - Primary key columns
    - Columns containing 'is_error', 'is_warning', or '_percentage'
    - Technical columns like 'created_at', 'updated_at', etc.

    Parameters:
    - all_columns: List of all column names in the table.
    - primary_keys: List of primary key columns (e.g., ['plant', 'material_code', 'valuation_class']).

    Returns:
    - List of additional column names to be checked for updates.
    """

    # Define keywords to exclude columns
    exclude_keywords = ["created_at", "updated_at", "deleted_at"]

    # Filter out primary key columns and columns that match the exclude conditions
    additional_columns = [
        col for col in all_columns
        if col not in primary_keys and not any(keyword in col for keyword in exclude_keywords)
    ]

    return additional_columns

In [0]:
def handle_table_update(df, table_name, primary_keys, all_columns, additional_columns_to_check=None, mode="update"):
    """
    Handle data insertion/update for a Delta table with multiple modes (update/full).
    Supports complex types including STRUCT and ARRAY<STRUCT>.
    
    Parameters:
    - df: DataFrame containing the new data to be inserted or updated.
    - table_name: The name of the target Delta table.
    - primary_keys: List of primary key columns to check.
    - all_columns: List of all columns in the Delta table.
    - additional_columns_to_check: List of additional columns for updates (optional, will be auto-detected).
    - mode: Mode for operation: "update" for update mode, "full" for overwrite mode (default is "update").
    """
    
    def is_complex_type(df, col_name):
        """Check if a column is a complex type (STRUCT or ARRAY)"""
        col_type = dict(df.dtypes)[col_name]
        return col_type.startswith('struct') or col_type.startswith('array')
    
    def get_column_comparison(col_name, is_complex):
        """Generate comparison expression based on column type"""
        if is_complex:
        # For complex types (STRUCT/ARRAY), convert to JSON string for comparison
            return f"NOT (to_json(target.{col_name}) <=> to_json(source.{col_name}))"
        else:
        # For all simple types, use NULL-safe comparison
            return f"NOT (target.{col_name} <=> source.{col_name})"
    

    # If no additional columns are provided, fetch them dynamically
    if additional_columns_to_check is None:
        additional_columns_to_check = get_additional_columns(all_columns, primary_keys)

    # Add created_at or updated_at column based on mode
    if mode == "full":
        # For full mode, we first delete all rows in the table
        target_delta_table = DeltaTable.forName(spark, table_name)

        # Delete all rows in the table
        target_delta_table.delete()

        # Add created_at column to the dataframe
        df = df.withColumn("created_at", current_timestamp())
        
        # Insert the new data into the table
        df.write.format("delta").mode("append").saveAsTable(table_name)
        
        # Get the number of rows inserted
        num_rows_inserted = df.count()

        # Print the number of rows inserted
        print(f"Table {table_name} has been fully replaced (delete + insert).")
        print(f"Number of rows inserted: {num_rows_inserted}")
    
    elif mode == "update":
        # Read the target table to perform updates
        target_delta_table = DeltaTable.forName(spark, table_name)
        
        # Check the columns in the target table and source DataFrame
        target_columns = target_delta_table.toDF().columns
        source_columns = df.columns
        
        # Ensure that all primary key columns exist in both DataFrames
        for pk in primary_keys:
            if pk not in source_columns or pk not in target_columns:
                raise ValueError(f"Primary key column '{pk}' is missing in source or target DataFrame.")
        
        # Ensure that all additional columns to check for updates exist in the source
        for col in additional_columns_to_check:
            if col not in source_columns:
                raise ValueError(f"Column '{col}' is missing in the source DataFrame.")
        
        # Construct the join condition for primary keys
        primary_key_condition = " AND ".join([f"target.{pk} = source.{pk}" for pk in primary_keys])

        # Construct the update condition based on additional columns with complex type support
        update_conditions = []
        for col in additional_columns_to_check:
            is_complex = is_complex_type(df, col)
            update_conditions.append(get_column_comparison(col, is_complex))
        
        update_condition = " OR ".join(update_conditions)
        
        # Perform the MERGE operation to apply the updates and inserts
        merge_result = target_delta_table.alias("target").merge(
            df.alias("source"),
            primary_key_condition
        ).whenMatchedUpdate(
            condition=update_condition,
            set={col: f"source.{col}" for col in additional_columns_to_check} | {"updated_at": current_timestamp()}
        ).whenNotMatchedInsert(
            values={col: f"source.{col}" for col in df.columns} | {"created_at": current_timestamp()}
        ).execute()

        # Get the history of the Delta table to inspect the merge operation
        history = target_delta_table.history(1)  # Get the most recent operation

        # Extract the number of rows updated and inserted from the history log
        num_rows_updated = history.select("operationMetrics.numTargetRowsUpdated").collect()[0][0]
        num_rows_inserted = history.select("operationMetrics.numTargetRowsInserted").collect()[0][0]

        # Print the result of the update operation
        print(f"Table {table_name} has been updated successfully.")
        # Display the number of rows affected
        print(f"Rows Updated: {num_rows_updated}")
        print(f"Rows Inserted: {num_rows_inserted}")

    else:
        raise ValueError("Invalid mode specified. Use 'update' or 'full'.")